In [28]:
import json
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from collections import defaultdict
import json


In [29]:
A = ["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"],
    "TT": ["Tetanus"],
    "HepB": ["Hepatitis_B"],
    "Hib": ["Hib"],
    "IPV": ["Polio"],
    "OPV": ["Polio"],
    "DT": ["Diphtheria", "Tetanus"],
    "Td": ["Diphtheria", "Tetanus"],
    "DTwP": ["Diphtheria", "Tetanus", "Pertussis"],
    "DTwP-Hib": ["Diphtheria", "Tetanus", "Pertussis", "Hib"],
    "Penta": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib"],
    "Hexa": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"],
    "HPV": ["HPV"],
    "Rotavirus": ["Rotavirus"],
    "PCV": ["PCV"]
}

P = [
    "AJ_Vaccines",
    "BB_NCIPD",
    "China_National",
    "Bharat_Biotech",
    "Bilthoven",
    "Biological_E",
    "GSK",
    "Haffkine_Bio",
    "LG_Chem",
    "Merck_Sharp",
    "Panacea_Biotec",
    "PT_Bio",
    "Sanofi",
    "Serum_Institute",
    "Pfizer"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"],
    "TT": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "HepB": ["Serum_Institute", "LG_Chem"],
    "Hib": ["Serum_Institute"],
    "IPV": ["LG_Chem", "AJ_Vaccines", "Bilthoven", "Sanofi"],
    "OPV": ["Serum_Institute", "PT_Bio", "GSK", "Sanofi", "Panacea_Biotec", "China_National", "Bharat_Biotech", "Haffkine_Bio"],
    "DT": ["PT_Bio", "BB_NCIPD"],
    "Td": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "DTwP": ["Serum_Institute", "Biological_E"],
    "DTwP-Hib": ["Serum_Institute"],
    "Penta": ["Serum_Institute", "PT_Bio", "Biological_E", "LG_Chem", "Panacea_Biotec"],
    "Hexa": ["Sanofi"],
    "HPV": ["GSK", "Merck_Sharp", "China_National"],
    "Rotavirus": ["Serum_Institute", "GSK", "Bharat_Biotech"],
    "PCV": ["Serum_Institute", "GSK", "Pfizer"]
}



## Define values

In [30]:
# define constants
beta = 10.0  

tmin = 1
tmax = 10

max_tender_length = 5

unit = 1000

Δ = [i for i in range(1, max_tender_length + 1)]

# Generate time periods
T = [*range(tmin, tmax + 1)]

# Calculate delta values
delta = {t: (1 + 0.03) ** t for t in T}

#  Tender cost
g = {t: 1e8/unit for t in T}

# Cost of expanding capacity for each producer
gamma = {p: 1e8/unit for p in P}  

# Inventory holding cost
h = {v: 0.01 for v in V}  

F_time_set = []

for t in T:
    for tau in T:
        if tau >= t:
            if (tau - t + 1) in Δ:
                F_time_set.append((t, tau))






## Read in necessary data

In [31]:
# import start data
filename = "data/Starting_point.xlsx"
starting_points_file_F = pd.read_excel(filename, sheet_name="F_start")

starting_points_vect_F = [
    (row['Antigen'], (row['Starting'], row['Ending']))
    for _, row in starting_points_file_F.iloc[0:].iterrows()
]

# starting_points_file_I = pd.read_excel(filename, sheet_name="I_start")

# starting_points_vect_I = [
#     (row['Vaccine'], (row['Amount']))
#     for _, row in starting_points_file_I.iloc[1:].iterrows()
# ]

#scenario probabilities
with open('data/scenario_pair_probabilities_new.json', 'r') as f:
    probabilities = json.load(f)

# import results
# Define the path to the file
# file_path = 'social_surplus_base.json'
file_path = 'social_surplus_base.json'

# Load the JSON file
with open(file_path, 'r') as file:
    data = json.load(file)

### read in and transform price data to dict

In [32]:

# Load the Excel file to examine its structure
file_path = 'data/Vaccine_price_data.xlsx'
xlsx = pd.ExcelFile(file_path)

# Get all sheet names and skip the first two sheets
sheet_names = xlsx.sheet_names[2:]

# Create a nested dictionary with structure: vaccine[producer][year]
vaccine_dict = {}

for sheet in sheet_names:
    # Read each sheet
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Create a nested dictionary for each sheet
    sheet_dict = {}
    for _, row in df.iterrows():
        producer = row['Unnamed: 0'] if 'Unnamed: 0' in row else None
        if producer:
            # Initialize dictionary for each producer
            if producer not in sheet_dict:
                sheet_dict[producer] = {}

            # Populate year data
            for col in df.columns:
                if isinstance(col, int):  # Assuming year columns are integers
                    sheet_dict[producer][col] = row[col]

    # Add sheet's nested dictionary to vaccine dictionary
    vaccine_dict[sheet] = sheet_dict

# Displaying a small portion of the resulting nested dictionary structure
vaccine_dict_sample = {sheet: list(vaccine_dict[sheet].items()) for sheet in vaccine_dict}
modified_dict = {}

for key, value in vaccine_dict_sample.items():
    # Split the key by space and keep only the first part
    new_key = key.split()[0]
    # Add the new key with the original value to the new dictionary
    modified_dict[new_key] = value

# Replace the original dictionary with the modified one
vaccine_price_dict = modified_dict

for vaccine, producers_list in vaccine_price_dict.items():
    # Convert list of tuples to a dictionary
    producers_dict = dict(producers_list)
    # Replace the list with the newly created dictionary
    vaccine_price_dict[vaccine] = producers_dict


## Calculate average price per year per vaccine

In [33]:
# Calculate average cost of vaccine per year
avg_prices_per_period = {}

for vaccine, producers in vaccine_price_dict.items():
    avg_prices_per_period[vaccine] = {}
    
    # Collect prices by time period
    prices_by_time = {}
    for producer, values in producers.items():
        for time, price in values.items():
            # Collecting prices properly, ensuring the value is a number
            if isinstance(price, (int, float)):
                if time not in prices_by_time:
                    prices_by_time[time] = []
                prices_by_time[time].append(price)

    # Calculate average price for each time period
    for time, prices in prices_by_time.items():
        avg_prices_per_period[vaccine][time] = sum(prices) / len(prices) if prices else 0

# avg_prices_per_period

## Calculate Tender costs - g[t] * F[a,t,tau] / delta[t] - working

In [34]:
# Initialize the results dictionary
result = {}

# Calculate the formula
for antigen, data_t in data.get("F", {}).items():  # Iterate over antigens
    result[antigen] = {}
    for t, data_tau in data_t.items():  # Iterate over start times (t)
        t = int(t)  # Ensure `t` is treated as an integer
        if t in T:
            result[antigen][t] = {}
            for tau, value in data_tau.items():  # Iterate over end times (tau)
                result[antigen][t][tau] = g[t] * value / delta[t]

F_OBJ_Value = 0

for antigen, data_t in result.items():
    for t, data_tau in data_t.items():
        F_OBJ_Value += sum(data_tau.values())

F_OBJ_Value

# Display the total sum for all antigens
print(f"F Objective cost: {F_OBJ_Value}")


F Objective cost: 3166681.8903872883


## Calculate Capacity Extension Costs - gamma[p] * L[p,t] / delta[t] - WORKING but check

In [35]:
# Assuming the "L" key contains the data you mentioned
L_data = data.get("L", {})

transformed_L_data = {}

# Iterate through each producer in L_data
for producer, years in L_data.items():
    for year, value in years.items():
        # If the year doesn't exist in transformed_L_data, initialize it as an empty dictionary
        if year not in transformed_L_data:
            transformed_L_data[year] = {}
        
        # Set the value for the producer in the corresponding year
        transformed_L_data[year][producer] = value

# The transformed_L_data now has the structure year[producer][value]
# print(transformed_L_data)


In [36]:
# Create a new dictionary to store the results after multiplication
result_after_gamma = {}

# Iterate through each year in transformed_L_data
for year, producers in transformed_L_data.items():
    result_after_gamma[year] = {}
    
    # Iterate through each producer in the year
    for producer, value in producers.items():
        # Multiply the producer's value by the corresponding value from gamma
        result_after_gamma[year][producer] = value * gamma[producer] / delta[int(year)]

# Initialize the total sum variable
L_OBJ_Value = 0

# Iterate through each year in result_after_gamma
for year, producers in result_after_gamma.items():
    # Iterate through each producer in the year and add its value to the total sum
    for producer, value in producers.items():
        L_OBJ_Value += value

# The total_sum variable will contain the sum of all the values
print(f"L Objective Value: {L_OBJ_Value}")



L Objective Value: 7447220.395631273


## Calculate missed doses by scenario - Beta * S[a,t,omega] / delta[t] - WORKING

In [37]:
def process_scenarios(S_data, beta, delta):
    # Step 1: Reorganize data with scenario at the top level
    S_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for antigen, year_data in S_data.items():
        for year, scenario_data in {y: d for y, d in year_data.items() if y != '0'}.items():
            for scenario, value in scenario_data.items():
                S_data_by_scenario[scenario][year][antigen] = value

    # Convert to regular dictionary
    S_data_by_scenario = dict(S_data_by_scenario)

    # Step 2: Scale data
    S_data_by_scenario_scaled = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for scenario, years in S_data_by_scenario.items():
        for year, antigens in years.items():
            for antigen in antigens.keys():
                S_data_by_scenario_scaled[scenario][year][antigen] = (
                    S_data_by_scenario[scenario][year][antigen] * beta / delta[int(year)]
                )

    # Step 3: Calculate scenario sums
    scenario_sums = {}
    for scenario, years in S_data_by_scenario_scaled.items():
        scenario_sum = sum(
            value
            for year in years.values()
            for value in year.values()
            if isinstance(value, (int, float))
        )
        scenario_sums[scenario] = scenario_sum

    return scenario_sums

In [38]:
S_data = data['S']
scenario_sums_S = process_scenarios(S_data, beta, delta)

In [39]:
S_OBJ_Values = {k: probabilities[k] * scenario_sums_S[k] for k in probabilities}
S_OBJ_Value = sum(S_OBJ_Values.values())
# S_OBJ_Value
print(f"Missed Dose OBJ Value: {(S_OBJ_Value)}")

Missed Dose OBJ Value: 151975068.6799311


## Calculate doses purchased - r[v,p,t] * X[v,p,t,omega] / delta[t] - WORKING

In [40]:
X_data = data['X']
# Reorganize X_data to make `omega` the first key while keeping the rest of the structure
def reorganize_x_data(x_data):
    reorganized_x_data = {}
    for v, producers in x_data.items():
        for p, time_periods in producers.items():
            for t, scenarios in time_periods.items():
                for omega, value in scenarios.items():
                    if omega not in reorganized_x_data:
                        reorganized_x_data[omega] = {}
                    if v not in reorganized_x_data[omega]:
                        reorganized_x_data[omega][v] = {}
                    if p not in reorganized_x_data[omega][v]:
                        reorganized_x_data[omega][v][p] = {}
                    reorganized_x_data[omega][v][p][t] = value
    return reorganized_x_data

# Reorganize X_data by scenario
reorganized_x_data = reorganize_x_data(X_data)

In [52]:
def calculate_scenario_results(reorganized_x_data, vaccine_price_dict, delta):
    scenario_results = {}
    for omega, vaccines in reorganized_x_data.items():
        scenario_results[omega] = {}
        for v, producers in vaccines.items():
            if v in vaccine_price_dict:
                for p, time_periods in producers.items():
                    if p in vaccine_price_dict[v]:
                        for t, value in time_periods.items():
                            t_int = int(t)
                            if t_int in vaccine_price_dict[v][p]:
                                scenario_results[omega][(v, p, t)] = (
                                    vaccine_price_dict[v][p][t_int] * value
                                ) / delta[t_int]
    return scenario_results

# Calculate scenario results
scenario_results = calculate_scenario_results(reorganized_x_data, vaccine_price_dict, delta)

In [42]:
X_OBJ_Values = {k: probabilities[k] * scenario_results[k] for k in probabilities}
X_OBJ_Value = sum(X_OBJ_Values.values())
X_OBJ_Value
print(f"Vaccines Purchased OBJ Value: {X_OBJ_Value}")

Vaccines Purchased OBJ Value: 19216003.373250175


## calculate inventory holding costs -h[v] * r_bar[v,t] * I[v,t,omega] / delta[t] 

In [43]:
I_data = data.get("I", {})

reversed_data = defaultdict(lambda: defaultdict(dict))
for vaccine, years in I_data.items():
    for year, scenarios in years.items():
            for scenario, value in scenarios.items():
                reversed_data[scenario][year][vaccine] = value 

In [44]:
def remove_year_zero(data):
    for scenario, years in list(data.items()):
        if isinstance(years, defaultdict):  # Ensure it's a defaultdict or dict
            for year in list(years.keys()):
                if year == '0':  # Check if the year is 0 (string form)
                    del years[year]  # Remove the entry
    return data

reversed_remove_start_I = remove_year_zero(reversed_data)

In [45]:
result = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))

for scenario, years in reversed_remove_start_I.items():
    for year, vaccines in years.items():
        for vaccine, value in vaccines.items():
                result[scenario][year][vaccine] = (value * avg_prices_per_period[vaccine][int(year)] * h[vaccine]) / delta[int(year)]

# The "result" dictionary will contain the multiplied values.


In [46]:
scenario_sums_H = {}

# Iterate through the `result` dictionary to calculate the sum by scenario.
for scenario, years in result.items():
    scenario_sum = 0  # Initialize the sum for the scenario.
    for year, vaccines in years.items():
        for vaccine, value in vaccines.items():
            scenario_sum += value  # Add the value to the sum for the scenario.
    
    scenario_sums_H[scenario] = scenario_sum  # Store the calculated sum for each scenario.

# `scenario_sums` will contain the total sum for each scenario.
print(scenario_sums_H)

{'5': 132763.17815905134, '35': 120480.29609671669, '16': 127578.08455263835, '20': 130812.50476973777, '12': 128111.60051220184, '30': 120857.20504137699, '24': 127336.4801992433, '8': 133131.9672638335, '28': 127000.93994872687, '17': 133330.0722218535, '1': 120317.83979093941, '23': 141584.62123782892, '32': 141879.09064960256, '22': 132918.19941885857, '6': 130645.3732876029, '19': 120089.46677740532, '11': 132198.34056472612, '31': 127723.65521369378, '9': 127383.56306881302, '14': 135832.85488879084, '3': 141217.71556760493, '29': 130893.79631944888, '33': 133257.5631766542, '7': 141388.3037929611, '25': 121844.80445678391, '34': 130622.78970215509, '13': 126951.12089913353, '4': 126697.42496597106, '15': 123532.64955248372, '2': 131064.49034856803, '10': 120853.44100523613, '27': 141378.99787589654, '26': 133060.40471337413, '18': 141384.0980106803, '21': 130529.27555429656}


In [47]:
I_OBJ_Values = {k: probabilities[k] * scenario_sums_H[k] for k in probabilities}
I_OBJ_Value = sum(I_OBJ_Values.values())
print(f"Inventory Holding OBJ Value: {(I_OBJ_Value)}")

Inventory Holding OBJ Value: 132455.6975694488


# TOTAL OBJ VALUE

In [48]:
print(f"OBJ Value: {F_OBJ_Value + L_OBJ_Value + X_OBJ_Value + S_OBJ_Value + I_OBJ_Value}")

OBJ Value: 181937430.0367693


In [49]:
print(f"OBJ Value: {S_OBJ_Value}")

OBJ Value: 151975068.6799311
